In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

import joblib
from pathlib import Path

In [2]:
train_df = pd.read_csv("../data/UNSW_NB15_training-set.csv")
test_df = pd.read_csv("../data/UNSW_NB15_testing-set.csv")

print("Training:", train_df.shape)
print("Testing:", test_df.shape)

Training: (175341, 45)
Testing: (82332, 45)


In [3]:
LIVE_FEATURES = [
    "dur",
    "proto",
    "spkts",
    "dpkts",
    "sbytes",
    "dbytes",
    "rate",
    "sinpkt",
    "dinpkt",
    "smean",
    "dmean"
]

X_train_live = train_df[LIVE_FEATURES]
y_train_live = train_df["label"]

X_test_live = test_df[LIVE_FEATURES]
y_test_live = test_df["label"]

print(X_train_live.head())
print("\nFeatures:", len(LIVE_FEATURES))

        dur proto  spkts  dpkts  sbytes  dbytes       rate      sinpkt  \
0  0.121478   tcp      6      4     258     172  74.087490   24.295600   
1  0.649902   tcp     14     38     734   42014  78.473372   49.915000   
2  1.623129   tcp      8     16     364   13186  14.170161  231.875571   
3  1.681642   tcp     12     12     628     770  13.677108  152.876547   
4  0.449454   tcp     10      6     534     268  33.373826   47.750333   

       dinpkt  smean  dmean  
0    8.375000     43     43  
1   15.432865     52   1106  
2  102.737203     46    824  
3   90.235726     52     64  
4   75.659602     53     45  

Features: 11


In [4]:
categorical_live = ["proto"]

numeric_live = [
    feature
    for feature in LIVE_FEATURES
    if feature not in categorical_live
]

preprocessor_live = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_live
        ),
        (
            "numeric",
            "passthrough",
            numeric_live
        )
    ]
)

live_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_live
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=150,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced"
            )
        )
    ]
)

print("Live model pipeline created!")

Live model pipeline created!


In [5]:
live_model.fit(
    X_train_live,
    y_train_live
)

print("Live model trained!")

Live model trained!


In [6]:
live_predictions = live_model.predict(
    X_test_live
)

print(
    classification_report(
        y_test_live,
        live_predictions,
        target_names=[
            "Normal",
            "Attack"
        ]
    )
)

print(
    confusion_matrix(
        y_test_live,
        live_predictions
    )
)

print(
    "Accuracy:",
    accuracy_score(
        y_test_live,
        live_predictions
    )
)

              precision    recall  f1-score   support

      Normal       0.94      0.81      0.87     37000
      Attack       0.86      0.95      0.90     45332

    accuracy                           0.89     82332
   macro avg       0.90      0.88      0.89     82332
weighted avg       0.89      0.89      0.89     82332

[[29947  7053]
 [ 2066 43266]]
Accuracy: 0.8892411213137055


In [7]:
Path("../models").mkdir(
    exist_ok=True
)

joblib.dump(
    live_model,
    "../models/live_intrusion_model.joblib"
)

print(
    "Live intrusion model saved!"
)

Live intrusion model saved!
